In [ ]:
import os
import pandas as pd
import shutil
from sqlalchemy import create_engine
import unicodedata
import warnings

# Ignorar todos los warnings
warnings.filterwarnings("ignore")

In [38]:
fechamin = '2024-06-01'
fechamax = '2024-06-30'
sql = "SELECT xDate,"
sql += "EOMONTH(xDate) AS EndOfMonth,"
sql += "CostCenterNo as centro_costo, "
sql += "ComplexEntityNo as granja_lote,"
sql += "SystemCostElementNo as elemento_costo,"
sql += "SystemLocationGroupNo as grupo_ubicacion,"
sql += "RelativeAmount as cantidad_relativa,"
sql += "SystemCostObjectNo as objeto_costo,"
sql += "SourceCode as codigo_fuente,"
sql += "SystemStageNo as etapa_sistema,"
sql += "RelativeUnits as unidades_relativas"
sql += " FROM mtech.mvProteinJournalTrans"
sql += f" WHERE mvProteinJournalTrans.IRN IS NOT NULL"
sql += f" AND mvProteinJournalTrans.xDate BETWEEN '{fechamin}' AND '{fechamax}'"
sql += f" AND mvProteinJournalTrans.SystemLocationGroupNo LIKE 'HAT %'"
sql += f" AND mvProteinJournalTrans.SourceCode IN ('EOP B-HIM - Hatcher', 'EOP B-HIM - Allocations')"
sql += f" AND mvProteinJournalTrans.SystemStageNo LIKE 'CHXPLT%';"

print(sql)

SELECT xDate,EOMONTH(xDate) AS EndOfMonth,CostCenterNo as centro_costo, ComplexEntityNo as granja_lote,SystemCostElementNo as elemento_costo,SystemLocationGroupNo as grupo_ubicacion,RelativeAmount as cantidad_relativa,SystemCostObjectNo as objeto_costo,SourceCode as codigo_fuente,SystemStageNo as etapa_sistema,RelativeUnits as unidades_relativas FROM mtech.mvProteinJournalTrans WHERE mvProteinJournalTrans.IRN IS NOT NULL AND mvProteinJournalTrans.xDate BETWEEN '2024-06-01' AND '2024-06-30' AND mvProteinJournalTrans.SystemLocationGroupNo LIKE 'HAT %' AND mvProteinJournalTrans.SourceCode IN ('EOP B-HIM - Hatcher', 'EOP B-HIM - Allocations') AND mvProteinJournalTrans.SystemStageNo LIKE 'CHXPLT%';


In [13]:
import pyodbc as pc
import pandas as pd

def connect_to_sql_server(server, database, username, password):
  """
  Establishes a secure connection to a SQL Server database using pyodbc.

  Args:
      server (str): The name or IP address of the SQL Server instance.
      database (str): The name of the database to connect to.
      username (str): The username for the database connection.
      password (str): The password for the database connection.

  Returns:
      pyodbc.connect object: A connection object to the SQL Server database.

  Raises:
      pyodbc.Error: If an error occurs while connecting to the database.
  """

  try:
    connection_string = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes;'
      # Connect securely using password parameter in connect()
    print(connection_string)
    conn = pc.connect(connection_string, password=password)
    return conn

  except pc.Error as ex:
    print("Error connecting to SQL Server:", ex)
    return None  # Indicate connection failure

In [39]:
server = '192.168.3.26'
database = 'Mtech_v7'
username = 'PRONACA\genapi'
password = 'Inverna2k20'

# Establish a secure connection
conn = connect_to_sql_server(server, database, username, password)

if conn:
    
    # Proceed with database operations using conn
    # ... (your SQL queries and data processing)
    # Ejecuta la consulta armada previamente en la variable sql
    cursor = conn.cursor()
    query = sql if 'sql' in globals() else "SELECT TOP 5 TABLE_NAME FROM INFORMATION_SCHEMA.TABLES"
    try:
        cursor.execute(sql)
    except Exception as ex:
        print(f"Primary query failed: {ex}")
        cursor.execute("SELECT TOP 5 TABLE_SCHEMA, TABLE_NAME FROM INFORMATION_SCHEMA.TABLES")

    resultados = cursor.fetchall()
    columnas = [col[0] for col in cursor.description] if cursor.description else []

    # Imprime los resultados
    for row in resultados:
        print(row)


    # Close the connection when finished
    conn.close()
    print("Connection closed successfully.")
else:
    print("Connection failed. Please check your credentials and network settings.")


DRIVER={ODBC Driver 17 for SQL Server};SERVER=192.168.3.26;DATABASE=Mtech_v7;Trusted_Connection=yes;
(datetime.datetime(2024, 6, 30, 0, 0), datetime.date(2024, 6, 30), '667101              ', 'AH-2723', 'FLETES              ', 'HAT                 ', Decimal('1109.1965000000'), 'OVHD                ', 'EOP B-HIM - Allocations                 ', 'CHXPLT              ', Decimal('152230.0000000000'))
(datetime.datetime(2024, 6, 30, 0, 0), datetime.date(2024, 6, 30), '667101              ', 'TO-1323', 'FLETES              ', 'HAT                 ', Decimal('1101.8373000000'), 'OVHD                ', 'EOP B-HIM - Allocations                 ', 'CHXPLT              ', Decimal('151220.0000000000'))
(datetime.datetime(2024, 6, 30, 0, 0), datetime.date(2024, 6, 30), '667101              ', 'AH-2123', 'FLETES              ', 'HAT                 ', Decimal('1047.0442000000'), 'OVHD                ', 'EOP B-HIM - Allocations                 ', 'CHXPLT              ', Decimal('143700.0000000000'))

In [40]:
# Convertir los resultados en un DataFrame de Pandas
if 'resultados' in globals() and 'columnas' in globals() and len(resultados) > 0:
    filas = [tuple(r) if not isinstance(r, tuple) else r for r in resultados]
    if len(columnas) == len(filas[0]):
        df = pd.DataFrame.from_records(filas, columns=columnas)
    else:
        df = pd.DataFrame.from_records(filas)
    # Procesar el DataFrame
    print(df.head())  # Mostrar las primeras filas del DataFrame
else:
    print("No hay resultados para convertir a DataFrame. Ejecuta primero la celda 4.")

       xDate  EndOfMonth          centro_costo granja_lote  \
0 2024-06-30  2024-06-30  667101                   AH-2723   
1 2024-06-30  2024-06-30  667101                   TO-1323   
2 2024-06-30  2024-06-30  667101                   AH-2123   
3 2024-06-30  2024-06-30  667101                   TO-1723   
4 2024-06-30  2024-06-30  667101                   AH-2523   

         elemento_costo       grupo_ubicacion cantidad_relativa  \
0  FLETES                HAT                    1109.1965000000   
1  FLETES                HAT                    1101.8373000000   
2  FLETES                HAT                    1047.0442000000   
3  FLETES                HAT                      55.3760000000   
4  FLETES                HAT                    1113.3497000000   

           objeto_costo                             codigo_fuente  \
0  OVHD                  EOP B-HIM - Allocations                    
1  OVHD                  EOP B-HIM - Allocations                    
2  OVHD          

In [37]:
server_destino = '1sq-pipbdd'
database_destino = 'Mtech_v7'
username_destino = 'PRONACA\genapi'
password_destino = 'Inverna2k20'